In [1]:
import requests
import pandas as pd
import time
from datetime import datetime, timedelta

url = "https://pncp.gov.br/api/consulta/v1/contratacoes/publicacao"
cnpj_mme = "37115383000153"
modalidades = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]

data_inicio = datetime(2020, 1, 1)
data_fim_periodo = datetime(2025, 12, 31)

todos_contratos = []
print(f"--- Extração PNCP para CNPJ {cnpj_mme} ---\n")

data_atual = data_inicio

while data_atual <= data_fim_periodo:
    proximo_mes = (data_atual.replace(day=28) + timedelta(days=4)).replace(day=1)
    fim_mes = proximo_mes - timedelta(days=1)
    
    str_inicio = data_atual.strftime("%Y%m%d")
    str_fim = fim_mes.strftime("%Y%m%d")
    
    print(f"Período: {str_inicio} - {str_fim}")

    for modalidade in modalidades:
        pagina = 1
        while True:
            params = {
                "dataInicial": str_inicio,
                "dataFinal": str_fim,
                "codigoModalidadeContratacao": modalidade,
                "cnpj": cnpj_mme,
                "pagina": pagina,
                "tamanhoPagina": 50
            }

            response = requests.get(url, params=params)
            
            if response.status_code == 200:
                conteudo = response.json()
                dados_pagina = conteudo.get('data', [])
                
                if dados_pagina:
                    todos_contratos.extend(dados_pagina)
                    print(f"  Modalidade {modalidade}: {len(dados_pagina)} itens (pág {pagina})")
                    
                    if len(dados_pagina) < 50:
                        break
                    pagina += 1
                    time.sleep(0.3)
                else:
                    break
            elif response.status_code == 404:
                break
            else:
                print(f"  Modalidade {modalidade}: Erro {response.status_code} - {response.text[:100]}")
                break
    
    data_atual = proximo_mes
    time.sleep(0.5)

if todos_contratos:
    df_resultado = pd.DataFrame(todos_contratos)
    print(f"\n--- Total extraído: {len(df_resultado)} registros ---")
    print(df_resultado.head())
else:
    print("\nNenhum dado encontrado.")

--- Extração PNCP para CNPJ 37115383000153 ---

Período: 20200101 - 20200131
  Modalidade 1: Erro 204 - 
  Modalidade 2: Erro 204 - 
  Modalidade 3: Erro 204 - 
  Modalidade 4: Erro 204 - 
  Modalidade 5: Erro 204 - 
  Modalidade 6: Erro 204 - 
  Modalidade 7: Erro 204 - 
  Modalidade 8: Erro 204 - 
  Modalidade 9: Erro 204 - 
  Modalidade 10: Erro 204 - 
  Modalidade 11: Erro 204 - 
  Modalidade 12: Erro 204 - 
  Modalidade 13: Erro 204 - 
Período: 20200201 - 20200229
  Modalidade 1: Erro 204 - 
  Modalidade 2: Erro 204 - 
  Modalidade 3: Erro 204 - 
  Modalidade 4: Erro 204 - 
  Modalidade 5: Erro 204 - 


KeyboardInterrupt: 

In [ ]:
df_resultado.to_csv("/Users/lucasborges/Desktop/supply_chain/data/bronze/raw_contratacoes.csv", index=False)

df_resultado.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 522 entries, 0 to 521
Data columns (total 34 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   dataAtualizacao                    522 non-null    object 
 1   orgaoEntidade                      522 non-null    object 
 2   anoCompra                          522 non-null    int64  
 3   sequencialCompra                   522 non-null    int64  
 4   numeroCompra                       522 non-null    object 
 5   processo                           522 non-null    object 
 6   objetoCompra                       522 non-null    object 
 7   orgaoSubRogado                     0 non-null      object 
 8   unidadeOrgao                       522 non-null    object 
 9   unidadeSubRogada                   0 non-null      object 
 10  valorTotalHomologado               409 non-null    float64
 11  srp                                522 non-null    bool   

In [ ]:
import requests
import pandas as pd
import time
from datetime import datetime, timedelta

# Contratos firmados (com dados dos fornecedores)

url_contratos = "https://pncp.gov.br/api/consulta/v1/contratos"
cnpj_mme = "37115383000153"

data_inicio = datetime(2020, 1, 1)
data_fim_periodo = datetime(2025, 12, 31)

todos_contratos_fornecedores = []
print(f"--- Extração de CONTRATOS (com fornecedores) para CNPJ {cnpj_mme} ---\n")

data_atual = data_inicio

while data_atual <= data_fim_periodo:
    proximo_mes = (data_atual.replace(day=28) + timedelta(days=4)).replace(day=1)
    fim_mes = proximo_mes - timedelta(days=1)
    
    str_inicio = data_atual.strftime("%Y%m%d")
    str_fim = fim_mes.strftime("%Y%m%d")
    
    print(f"Período: {str_inicio} - {str_fim}")
    
    pagina = 1
    while True:
        params = {
            "dataInicial": str_inicio,
            "dataFinal": str_fim,
            "cnpjOrgao": cnpj_mme,
            "pagina": pagina,
            "tamanhoPagina": 500
        }

        response = requests.get(url_contratos, params=params)
        
        if response.status_code == 200:
            conteudo = response.json()
            dados_pagina = conteudo.get('data', [])
            
            if dados_pagina:
                todos_contratos_fornecedores.extend(dados_pagina)
                print(f"  {len(dados_pagina)} contratos (pág {pagina})")
                
                if len(dados_pagina) < 500:
                    break
                pagina += 1
                time.sleep(0.3)
            else:
                break
        elif response.status_code == 404:
            break
        else:
            print(f"  Erro {response.status_code} - {response.text[:100]}")
            break
    
    data_atual = proximo_mes
    time.sleep(0.5)

if todos_contratos_fornecedores:
    df_contratos = pd.DataFrame(todos_contratos_fornecedores)
    print(f"\n--- Total de contratos extraídos: {len(df_contratos)} registros ---")
    print(f"Colunas de fornecedor disponíveis: {[c for c in df_contratos.columns if 'fornecedor' in c.lower() or c == 'niFornecedor']}")
else:
    print("\nNenhum contrato encontrado.")

--- Extração de CONTRATOS (com fornecedores) para CNPJ 37115383000153 ---

Período: 20200101 - 20200131
  Erro 204 - 
Período: 20200201 - 20200229
  Erro 204 - 
Período: 20200301 - 20200331
  Erro 204 - 
Período: 20200401 - 20200430
  Erro 204 - 
Período: 20200501 - 20200531
  Erro 204 - 
Período: 20200601 - 20200630
  Erro 204 - 
Período: 20200701 - 20200731
  Erro 204 - 
Período: 20200801 - 20200831
  Erro 204 - 
Período: 20200901 - 20200930
  Erro 204 - 
Período: 20201001 - 20201031
  Erro 204 - 
Período: 20201101 - 20201130
  Erro 204 - 
Período: 20201201 - 20201231
  Erro 204 - 
Período: 20210101 - 20210131
  Erro 204 - 
Período: 20210201 - 20210228
  Erro 204 - 
Período: 20210301 - 20210331
  Erro 204 - 
Período: 20210401 - 20210430
  Erro 204 - 
Período: 20210501 - 20210531
  Erro 204 - 
Período: 20210601 - 20210630
  Erro 204 - 
Período: 20210701 - 20210731
  Erro 204 - 
Período: 20210801 - 20210831
  Erro 204 - 
Período: 20210901 - 20210930
  2 contratos (pág 1)
Período: 20211

In [ ]:
# Salvar contratos com dados de fornecedores
df_contratos.to_csv("/Users/lucasborges/Desktop/supply_chain/data/bronze/raw_contratos.csv", index=False)

print("Colunas principais de fornecedor:")
print(f"  - niFornecedor (CNPJ/CPF): {df_contratos['niFornecedor'].nunique()} fornecedores únicos")
print(f"  - nomeRazaoSocialFornecedor: exemplos abaixo")
print(df_contratos[['niFornecedor', 'nomeRazaoSocialFornecedor', 'valorInicial']].head(10))

df_contratos.info()

Colunas principais de fornecedor:
  - niFornecedor (CNPJ/CPF): 174 fornecedores únicos
  - nomeRazaoSocialFornecedor: exemplos abaixo
     niFornecedor                          nomeRazaoSocialFornecedor  \
0  10918347000171                         DIAGRAMA TECNOLOGIA EIRELI   
1  10719671000160     ELDEX DISTRIBUIDORA DE JORNAIS E REVISTAS LTDA   
2  13728507000108           SILVIO APARECIDO DE MEDEIROS ELETRONICOS   
3  14911164000185                          AMV FESTAS & EVENTOS LTDA   
4  01319181000186       LAVANDERIA CRISTAL SERVICOS EXPRESSOS EIRELI   
5  37994340000195  TECNOFOTO COMERCIO DE EQUIPAMENTOS ELETRONICOS...   
6  35600597000190                VS PLANEJAMENTO E CONSTRUCAO EIRELI   
7  36895360000146                ICARO RODRIGUES MEIRINO 00269943218   
8  26673492000170           BRAULIO VINICIUS CARDOSO DE SOUZA EIRELI   
9  26673492000170           BRAULIO VINICIUS CARDOSO DE SOUZA EIRELI   

   valorInicial  
0       5967.00  
1       5447.24  
2       2079.60  
3

In [ ]:
# Cadastro de Empresas Inidôneas - CEIS/CNEP
# Período de publicação: 01/01/2024 a 31/12/2025

fornecedores_inidoneos = pd.read_csv("/Users/lucasborges/Desktop/supply_chain/data/bronze/raw_sancoes.csv", sep=";")

In [ ]:
fornecedores_inidoneos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5485 entries, 0 to 5484
Data columns (total 10 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   CNPJ/CPF da Pessoa ou Empresa Sancionada  5485 non-null   object 
 1   Nome da Pessoa ou Empresa Sancionada      5485 non-null   object 
 2   Cadastro                                  5485 non-null   object 
 3   UF sancionado                             5485 non-null   object 
 4   Nome do Órgão Sancionador                 5485 non-null   object 
 5   Categoria sanção                          5485 non-null   object 
 6   Data Publicação                           5485 non-null   object 
 7   Valor multa                               5485 non-null   object 
 8   Quantidade                                5485 non-null   int64  
 9   Unnamed: 9                                0 non-null      float64
dtypes: float64(1), int64(1), object(8)
m

In [ ]:
pip install python-bcb

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Índices Econômicos

from bcb import sgs

# O uso é idêntico ao que você queria
indices = sgs.get({'IPCA': 433, 'IGPM': 189}, start='2024-01-01', end='2025-12-31')
indices.info()

indices = indices.to_csv("/Users/lucasborges/Desktop/supply_chain/data/bronze/raw_indices.csv", index=False)

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 24 entries, 2024-01-01 to 2025-12-01
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   IPCA    24 non-null     float64
 1   IGPM    24 non-null     float64
dtypes: float64(2)
memory usage: 576.0 bytes
